### Finde kurze Pfade

In [ ]:
import babycube as cube
import search_strategies as S
import heapq


def get_neighbors(state):
    for k, op in cube.KEY_OP.items():
        new_state = cube.apply_op(op, state)
        yield new_state


def get_path_extension2(path):
    '''path: Tuple der Form (word_als_tuple, state_der_erreicht_wird)
       e.g. (('U', 'R'),  cube.apply_word('UR'))
    '''
    word, state = path
    for k, op in cube.KEY_OP.items():
        if len(word) > 1 and k.upper()[0] == word[-1][0].upper():
            continue

        new_state = cube.apply_op(op, state)
        new_word = word + (k,)
        yield (new_word, new_state)


def search_smart_1(start, get_path_extension, h, goal, max_priority):
    '''bricht suche ab, falls h(path) > max_priority
       Idee: dist_dict[state] liefert die Disanz von state xum goal
             path = (word, state)
             priority[0] ist dist_dict.get(state, float('inf')) + len(word),
             also die kuerzeste noch moegliche Gesamtlaenge des Pfades.
             Ist diese groesser als max_priority, wird die Suche abgebrochen
    '''
    count = 0
    path = (('',), start)  # word plus state
    priority = (h(path), count)
    nodes_to_visit = [(priority, 0, path)]
    seen = {path[0]}

    while nodes_to_visit:
        priority, depth, path = heapq.heappop(nodes_to_visit)
        if priority[0] > max_priority:
            break
        if path[1] == goal:
            yield path[0]

        for path in get_path_extension(path):
            if path[0] in seen:
                continue

            seen.add(path[0])
            count += 1
            priority = (h(path)+depth+1, count)
            heapq.heappush(nodes_to_visit, (priority, depth+1, path))

Finde alle kürzesten Wörter, die zum gleiche Zustand führen, wie
das Wort `'RUrURU2rU2'`.  
```python
sune = 'RUrURU2rU2'
scramble = cube.apply_word(sune)
scramble  # (0, 1, 2, 3, 4, 5, 6, 7, 1, 1, 1, 0, 0, 0, 0, 0)

# Lösung: ['RUR2uR2URu','RUrURU2rU2','RF2rU2RFuF','RfUFR2fU2F','fU2FUfUFU2','U2RFuFRF2r','U2fR2FRuRF', 'uFRF2rU2RF']
```

**Unidirektionale Suche**:
- Erstellen einen Distanz-Dict und eine Heuristik:
    ```python
    _, _, dist_dict8 = S.search_bf(scramble, get_neighbors, None, max_depth=8)


    def h(path):
        return dist_dict8.get(path[-1], float('inf'))
        
    ```

- Benutze eine Variante von search_smart, die die Suche abbricht, falls die Priority einen bestimmten Wert übertrifft.  

In [ ]:
scramble = cube.apply_word('RUrURU2rU2')
dist_dict8 = S.search_bf(scramble, get_neighbors, None, max_depth=8)[-1]

In [ ]:
dist_dict8[cube.ID]

In [ ]:
def h(path):
    return dist_dict8.get(path[-1], float('inf'))


# finde Pfade mit bis Laenge 8
words = list(search_smart_1(cube.ID, get_path_extension2, h, scramble, 8))
words = [''.join(word) for word in words]
words, len(words)

In [ ]:
{cube.apply_word(word) == scramble for word in words}

In [ ]:
# finde Pfade mit bis Laenge 9
words = list(search_smart_1(cube.ID, get_path_extension2, h, scramble, 9))
words = [''.join(word) for word in words]
words, len(words)

In [ ]:
{cube.apply_word(word) == scramble for word in words}

***
**Bidirektionale Suche**:  
- Erstelle Distanz-Dicts `dd_ID` und `dd_scramble`, und finde die gemeinsamen Knoten.
  ```python
  _, _, dd_ID = S.search_bf(cube.ID, get_neighbors, None, max_depth=4)
  _, _, dd_scramble = S.search_bf(scramble, get_neighbors, None, max_depth=4)
  
  midpoints = set(dd_ID.keys()) & set(dd_goal.keys())
  
  ```
  <br>
- Suche wie oben für jeden `midpoint` Pfade von `cube.ID` nach `midpoint` und von `scramble` nach `midpoint`.

**Finde Mittelpunkte**

In [ ]:
scramble = cube.apply_word('rF2R2fRFUF')
midpoint, go_backs = S.search_bibf(cube.ID, get_neighbors, scramble)

mp_to_start = S.get_path_home(midpoint, go_backs[0])
mp_to_scramble = S.get_path_home(midpoint, go_backs[1])

depth_1, depth_2 = len(mp_to_start) - 1, len(mp_to_scramble) - 1
depth_1, depth_2

In [ ]:
dd_start = S.search_bf(cube.ID, get_neighbors, None, depth_1)[-1]
dd_scramble = S.search_bf(scramble, get_neighbors, None, depth_2)[-1]

In [ ]:
midpoints = list(set(dd_start) & set(dd_scramble))
len(midpoints)

**Suche alle kurzen Pfade zu einem Mittelpunkt, vom Start und vom Ziel aus.**

In [ ]:
mp = midpoints[2]
dd = S.search_bf(mp, get_neighbors, None, depth_1)[-1]

In [ ]:
def h(path):
    return dd.get(path[-1], float('inf'))


words = list(search_smart_1(cube.ID, get_path_extension2, h, mp, depth_1))
words = [''.join(word) for word in words]
words, len(words)

In [ ]:
words = list(search_smart_1(scramble, get_path_extension2, h, mp, depth_1))
words = [''.join(word) for word in words]
words, len(words)

In [ ]:
paths = ['RF2R2FR2' + 'rFUF', 'rF2R2fR2' + 'rFUF']
{cube.apply_word(word) == scramble for word in paths}